<a href="https://colab.research.google.com/github/cmauriciocrescente-upc/Dataset-en-Colab/blob/main/Analisis_Limpieza_Datos_Faltantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Clase 4 – Tratamiento de Datos Faltantes
### Notebook ordenado para Google Colab

## Índice
0. [Configuración inicial](#scrollTo=seccion0)
1. [Ejercicio base – Inspección de un DataFrame](#scrollTo=seccion1)
2. [Ejercicio extra – Limpieza de datos de Ventas (duplicados, texto, fechas)](#scrollTo=seccion2)
3. [Bloque 2 Práctico – Limpieza_Nulos_Parte1 (dataset tipo House Prices)](#scrollTo=seccion3)
4. [Bloque 3 Teórico/Práctico – Imputación avanzada (por grupo)](#scrollTo=seccion4)
5. [Bloque 4 Práctico – Limpieza_Nulos_Parte2 (Titanic)](#scrollTo=seccion5)
6. [Bonus – Detección rápida de outliers](#scrollTo=seccion6)
7. [Conclusión y contenido para el README del entregable](#scrollTo=seccion7)

> 💡 Tip: en Colab, si le pones el mismo título a una celda de texto (Markdown) que empieza con `#`, `##`, etc., aparece automáticamente en el índice lateral. No hace falta que los links de arriba funcionen para que Colab arme el índice: alcanza con los títulos.


<a name='seccion0'></a>
# 0. Configuración inicial

Acá importamos todas las librerías que vamos a usar en el resto del notebook.

In [ ]:
# --- Librerías de siempre para trabajar con datos ---
import pandas as pd            # manejo de tablas (DataFrames)
import numpy as np             # números, y sobre todo np.nan para representar "faltante"
import seaborn as sns          # gráficos lindos + datasets de ejemplo (trae Titanic y Iris incorporados)
import matplotlib.pyplot as plt  # motor de gráficos que usa seaborn por debajo

from sklearn.impute import SimpleImputer          # imputador simple (media/mediana/moda/constante)
from sklearn.base import BaseEstimator, TransformerMixin  # para crear nuestro propio imputador (GroupImputer)

# Esto hace que los gráficos se vean un poco más prolijos
sns.set_theme(style='whitegrid')


<a name='seccion1'></a>
# 1. Ejercicio base – Inspección de un DataFrame

Antes de imputar nada, hay que saber **leer** un DataFrame: qué columnas tiene, de qué tipo son, y cómo filtrar filas/columnas. Esto es la base para todo lo que viene después.

In [ ]:
# Armamos un DataFrame "a mano" a partir de un diccionario de Python.
# Cada clave del diccionario se convierte en una columna.
data_inicial = {
    'ID_Cliente': [1001, 1002, 1003, 1004, 1005, 1006],
    'Nombre': ['Ana', 'Carlos', 'Elena', 'Diego', 'Beatriz', 'A'],
    'Edad': [25, 34, 28, 45, 31, 32],
    'Monto_Compra': [150.50, 200.00, 50.25, 310.00, 99.99, 1.1]
}
df_manual = pd.DataFrame(data_inicial)

# .info() nos dice: cantidad de filas, cantidad de nulos por columna y el tipo de dato (dtype) de cada una.
# Es SIEMPRE el primer comando que conviene correr al recibir un dataset nuevo.
df_manual.info()


In [ ]:
# Mostramos el DataFrame completo
df_manual


In [ ]:
# --- Selección de columnas ---
# Con doble corchete [[...]] elegimos varias columnas y el resultado sigue siendo un DataFrame
nombres_y_montos = df_manual[['Nombre', 'Monto_Compra']]
nombres_y_montos


In [ ]:
# --- Filtrado de filas por condición (filtro booleano) ---
# Esto devuelve solo las filas donde Monto_Compra sea mayor a 100
compras_altas = df_manual[df_manual['Monto_Compra'] > 100]
compras_altas


In [ ]:
# --- Indexación con iloc (por posición) y loc (por etiqueta) ---
primeras_dos_filas = df_manual.iloc[0:2]        # las filas en las posiciones 0 y 1
dato_especifico = df_manual.loc[2, 'Nombre']    # el valor de la fila con índice 2, columna 'Nombre' -> 'Elena'

print(dato_especifico)
primeras_dos_filas


<a name='seccion2'></a>
# 2. Ejercicio extra – Limpieza de datos de Ventas

Este ejercicio no imputa nulos, pero es un excelente calentamiento porque toca otros problemas típicos de calidad de datos: **duplicados**, **texto inconsistente** (mayúsculas/espacios) y **fechas en formatos distintos**. Son problemas de *completitud/consistencia*, hermanos de los datos faltantes.

In [ ]:
# Dataset de ventas con errores intencionales: duplicados, fechas en formatos distintos,
# nombres de clientes con mayúsculas/minúsculas y espacios de más.
ventas_dirty_data = {
    'ID_Transaccion': [1, 2, 2, 4, 5, 6, 7, 8],
    'Fecha': ['2024-01-10', '11/01/2024', '11/01/2024', '2024-01-13', '2024.01.14',
              '15-Jan-2024', None, '2024-01-17'],
    'Cliente': [' Juan Perez ', 'MARIA GOMEZ', 'MARIA GOMEZ ', 'Pedro perez', 'Ana Lopez',
                'Carlos Ruiz', 'Laura Fernandez', '  juan perez'],
    'Monto_USD': ['150.50', '200.00', '200.00', '350.00', '0.00', '180.00', '500.50', '150.50'],
    'Estado': ['Completado', 'completado', 'completado', 'P_Pendiente', 'Completado',
               'CANCELADO', 'Completado', 'Completado']
}
df_ventas = pd.DataFrame(ventas_dirty_data)
df_ventas


In [ ]:
# --- PASO 1: Detección y eliminación de duplicados ---
# drop_duplicates() elimina filas que son EXACTAMENTE iguales en todas sus columnas.
# El .copy() evita el warning de pandas por estar modificando una "vista" del DataFrame original.
df_ventas = df_ventas.drop_duplicates().copy()
df_ventas


In [ ]:
# --- PASO 2: Normalización de texto ---
# .str.strip()  -> saca espacios sobrantes al principio/final
# .str.title()  -> pone "Cada Palabra Con Mayúscula" (para nombres propios)
# .str.upper()  -> pone TODO EN MAYÚSCULAS (para categorías tipo Estado)
df_ventas['Cliente'] = df_ventas['Cliente'].str.strip().str.title()
df_ventas['Estado'] = df_ventas['Estado'].str.strip().str.upper()
df_ventas


In [ ]:
# --- PASO 3: Corrección y normalización de fechas ---
# pd.to_datetime con format='mixed' intenta reconocer varios formatos de fecha a la vez
# (2024-01-10, 11/01/2024, 15-Jan-2024, etc.).
# errors='coerce' hace que, si una fecha no se puede interpretar (o es None), en vez de romper
# el código, la convierta en NaT (Not a Time = el "nulo" de las fechas en pandas).
df_ventas['Fecha_Limpia'] = pd.to_datetime(df_ventas['Fecha'], format='mixed', errors='coerce')
df_ventas


In [ ]:
# --- PASO 4: Casteo numérico ---
# Monto_USD llegó como texto (string). pd.to_numeric lo convierte a número.
# errors='coerce' funciona igual que antes: lo que no se pueda convertir, queda como NaN.
df_ventas['Monto_USD'] = pd.to_numeric(df_ventas['Monto_USD'], errors='coerce')

# --- PASO 5: Validación final ---
print('--- Dataset resultante procesado ---')
display(df_ventas)

print('\n--- Diagnóstico de tipos final (dtypes) ---')
print(df_ventas.dtypes)


In [ ]:
# --- Bonus: .unique() ---
# .unique() devuelve los valores distintos que aparecen en una columna, sin repetir.
# Sirve para detectar rápido inconsistencias (ej: "COMPLETADO" vs "Completado" antes de normalizar).
print('Valores únicos en Cliente:', df_ventas['Cliente'].unique())
print('Valores únicos en Estado:', df_ventas['Estado'].unique())
print('Valores únicos en ID_Transaccion:', df_ventas['ID_Transaccion'].unique())


<a name='seccion3'></a>
# 3. Bloque 2 Práctico – `Limpieza_Nulos_Parte1.ipynb`

**Consigna:** dataset tipo *House Prices* reducido a columnas relevantes.
1. Visualizar los nulos (`isnull().sum()` + `heatmap`).
2. Eliminar columnas con más del 50% de nulos.
3. Eliminar filas con nulos en columnas clave.
4. Imputar numéricos con la **mediana** y categóricos con el valor **más frecuente (moda)**.

> ⚠️ No tenía el CSV real de *House Prices* de Kaggle, así que uso el dataset sintético de productos (`data_imputacion`) que armamos en clase: tiene la misma estructura del ejercicio (nulos + una columna "clave" tipo `SalePrice`) y sirve para practicar exactamente los mismos 4 pasos. Más abajo dejo también, comentada, la línea para descargar el dataset real de Kaggle por si querés usarlo tal cual pide la consigna.

In [ ]:
# --- (Opcional) Descarga del dataset real de House Prices desde Kaggle ---
# Esto SOLO funciona en Colab con tu cuenta de Kaggle configurada (kaggle.json).
# Lo dejamos comentado para no romper la ejecución si no lo tenés configurado.

# import kagglehub
# path = kagglehub.dataset_download("yasserh/housing-prices-dataset")
# print("Path to dataset files:", path)
# !mv {path}/Housing.csv /content/
# df_house = pd.read_csv('/content/Housing.csv')


In [ ]:
# --- Dataset de trabajo (sustituto de House Prices, mismo tipo de problema) ---
# Creamos un dataset de 12 productos con nulos "naturales" en Precio_Unitario y Calificacion_Cliente.
data_imputacion = pd.DataFrame({
    'ID_Producto': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'Categoria': ['Electronica', 'Ropa', 'Electronica', 'Ropa', 'Alimentos', 'Electronica',
                  'Ropa', 'Alimentos', 'Electronica', 'Ropa', 'Alimentos', 'Electronica'],
    'Region': ['Norte', 'Sur', 'Norte', 'Este', 'Oeste', 'Norte', 'Sur', 'Este', 'Oeste',
               'Norte', 'Sur', 'Este'],
    'Ventas': [1500, 200, 1700, 300, 50, 1600, 250, 70, 1800, 220, 60, 1900],
    'Precio_Unitario': [50.0, 15.0, np.nan, 20.0, 5.0, 55.0, 18.0, 7.0, np.nan, 16.0, 6.0, 60.0],
    'Stock': [100, 200, 80, 150, 300, 90, 220, 310, 75, 160, 320, 85],
    'Calificacion_Cliente': [4.5, 3.8, 4.0, np.nan, 3.5, 4.8, 3.9, 3.0, 4.2, 3.7, np.nan, 4.9]
})

# Sembramos ALGUNOS nulos adicionales al azar en Ventas y Stock, para que el ejercicio
# de "más del 50% de nulos" y de imputación tenga más sentido (simula un dataset real sucio).
np.random.seed(42)  # fija la semilla para que el resultado sea siempre el mismo al re-ejecutar
data_imputacion.loc[data_imputacion.sample(frac=0.1).index, 'Ventas'] = np.nan
data_imputacion.loc[data_imputacion.sample(frac=0.1).index, 'Stock'] = np.nan

# Y unos outliers (valores absurdamente altos), para usarlos más abajo en la sección de bonus.
data_imputacion.loc[3, 'Ventas'] = 50000        # outlier en Ventas
data_imputacion.loc[7, 'Precio_Unitario'] = 500  # outlier en Precio_Unitario
data_imputacion.loc[10, 'Stock'] = 10000         # outlier en Stock

# Lo guardamos como CSV para simular que "lo leemos desde un archivo", igual que en la consigna.
data_imputacion.to_csv('data_imputacion.csv', index=False)

df_house = pd.read_csv('data_imputacion.csv')
df_house


In [ ]:
# --- Paso 1: Visualizar los nulos ---

# isnull() devuelve True/False por celda; .sum() cuenta cuántos True (nulos) hay por columna.
print("Cantidad de nulos por columna:")
print(df_house.isnull().sum())

# El heatmap pinta de un color las celdas nulas y de otro las que tienen dato.
# Sirve para "ver" de un vistazo dónde se concentran los faltantes.
plt.figure(figsize=(10, 6))
sns.heatmap(df_house.isnull(), cbar=False, cmap='viridis')
plt.title('Mapa de calor de valores nulos - Dataset de productos (equivalente a House Prices)')
plt.show()


In [ ]:
# --- Paso 2: Eliminar columnas con más del 50% de nulos ---

# thresh = cantidad MÍNIMA de datos NO nulos que tiene que tener una columna para sobrevivir.
# Si una columna tiene menos del 50% de datos válidos, se elimina.
umbral = len(df_house) * 0.5
df_house = df_house.dropna(axis=1, thresh=umbral)  # axis=1 -> opera sobre columnas

print(f"Columnas luego del filtro (umbral={umbral} filas no nulas mínimas):")
print(df_house.columns.tolist())


In [ ]:
# --- Paso 3: Eliminar filas con nulos en columnas clave ---

# 'Precio_Unitario' es nuestra columna "clave" (equivalente a SalePrice en House Prices):
# si no sabemos el precio, esa fila no sirve para el análisis, así que la eliminamos directamente
# (a diferencia de otras columnas, que sí vamos a imputar en el paso siguiente).
df_house = df_house.dropna(subset=['Precio_Unitario'])

print(f"Filas restantes después de eliminar nulos en Precio_Unitario: {len(df_house)}")


In [ ]:
# --- Paso 4: Imputación de los nulos restantes ---

# Separamos las columnas según su tipo de dato:
num_cols = df_house.select_dtypes(include=['float64', 'int64']).columns  # numéricas
cat_cols = df_house.select_dtypes(include=['object']).columns            # texto/categóricas

# Numéricas -> imputamos con la MEDIANA (más robusta que el promedio frente a outliers,
# que es justo lo que tenemos en Ventas y Stock).
for col in num_cols:
    if df_house[col].isnull().sum() > 0:
        df_house[col] = df_house[col].fillna(df_house[col].median())

# Categóricas -> imputamos con la MODA (el valor más frecuente), porque no tiene sentido
# calcular un "promedio" de categorías de texto.
for col in cat_cols:
    if df_house[col].isnull().sum() > 0:
        df_house[col] = df_house[col].fillna(df_house[col].mode()[0])

print("Nulos restantes por columna (debería ser todo 0):")
print(df_house.isnull().sum())
df_house


<a name='seccion4'></a>
# 4. Bloque 3 – Imputación avanzada (por grupo)

**Idea central:** en vez de imputar con un único valor "global" (una mediana para toda la columna), a veces conviene imputar según el **grupo** al que pertenece cada fila (ej: la edad esperada de un pasajero de 1ra clase no es la misma que la de uno de 3ra clase). Esto reduce el riesgo de "aplanar" la distribución real de los datos.

In [ ]:
# --- 4.1 Ejemplo mínimo: groupby() + transform() ---

# transform() aplica una función por grupo pero DEVUELVE un resultado del mismo largo que el
# DataFrame original (a diferencia de agg(), que resume). Por eso sirve para imputar in-place.
data = pd.DataFrame({
    'Ciudad': ['Madrid', 'Madrid', 'Barcelona', 'Barcelona', 'Madrid', 'Barcelona'],
    'Ingresos': [3000, np.nan, 4000, 4500, 3200, np.nan]
})

# Para cada fila con Ingresos nulo, lo completamos con el promedio de Ingresos DE SU MISMA Ciudad.
data['Ingresos'] = data.groupby('Ciudad')['Ingresos'].transform(lambda x: x.fillna(x.mean()))
data


### 4.2 Un imputador por grupo reutilizable (`GroupImputer`)

Hacemos lo mismo que arriba pero empaquetado como una clase compatible con scikit-learn (tiene `.fit()` y `.transform()`), para poder reusarla en cualquier dataset sin reescribir el código cada vez.

In [ ]:
class GroupImputer(BaseEstimator, TransformerMixin):
    """Imputa valores nulos usando la media o mediana calculada POR GRUPO
    (por ejemplo: la mediana de Ingresos, agrupando por Ciudad)."""

    def __init__(self, group_cols, target_cols, strategy='mean'):
        self.group_cols = group_cols      # columna(s) que definen el grupo (ej: 'Ciudad')
        self.target_cols = target_cols    # columna(s) a imputar (ej: ['Ingresos'])
        self.strategy = strategy          # 'mean' o 'median'
        self.mapping_ = {}                # acá guardamos, por columna, el valor calculado por grupo

    def fit(self, X, y=None):
        # "Aprende" los valores de imputación por grupo (sin modificar los datos todavía)
        X_df = pd.DataFrame(X)
        for col in self.target_cols:
            if self.strategy == 'mean':
                group_values = X_df.groupby(self.group_cols)[col].mean()
            elif self.strategy == 'median':
                group_values = X_df.groupby(self.group_cols)[col].median()
            else:
                raise ValueError("La estrategia debe ser 'mean' o 'median'")
            self.mapping_[col] = group_values
        return self

    def transform(self, X):
        # Aplica lo aprendido en fit() para rellenar los nulos
        X_df = X.copy()
        for col in self.target_cols:
            group_series = X_df[self.group_cols]
            impute_values = group_series.map(self.mapping_[col])  # valor del grupo correspondiente
            X_df[col] = X_df[col].fillna(impute_values)

            # Si un grupo entero no tenía datos (caso raro), usamos el promedio global como respaldo
            global_fill = self.mapping_[col].mean()
            X_df[col] = X_df[col].fillna(global_fill)
        return X_df


# --- Probamos la clase con el mismo ejemplo de Ciudad/Ingresos ---
data_prueba = pd.DataFrame({
    'Ciudad': ['Madrid', 'Madrid', 'Barcelona', 'Barcelona', 'Madrid', 'Barcelona'],
    'Ingresos': [3000, np.nan, 4000, 4500, 3200, np.nan]
})

imputer = GroupImputer(group_cols='Ciudad', target_cols=['Ingresos'], strategy='mean')
data_imputada = imputer.fit_transform(data_prueba)
data_imputada


### 4.3 Ejercicio de aplicación: consultas veterinarias

Un ejemplo con contexto real (medicina/veterinaria, como menciona la teoría): imputamos **Peso** y **Edad_Estimada** (numéricas) con SimpleImputer, y **Diagnostico** (categórica) con la moda. Después comparamos cómo cambia la distribución del Peso antes y después.

In [ ]:
df_vet = pd.DataFrame({
    'Nombre': ['Fido', 'Luna', 'Max', 'Mimi', 'Rocky', 'Bella', 'Toby'],
    'Animal': ['Perro', 'Gato', 'Perro', 'Gato', 'Perro', 'Gato', 'Perro'],
    'Fecha_Consulta': ['2025-10-01', '2025-10-02', '2025-10-02', '2025-10-03', '2025-10-03',
                       '2025-10-04', '2025-10-05'],
    'Peso': [20.5, 4.2, np.nan, 3.8, 25.0, np.nan, 18.0],
    'Edad_Estimada': [5, np.nan, 3, 2, 6, 4, np.nan],
    'Diagnostico': ['Otitis', 'Gastroenteritis', 'Dermatitis', np.nan, 'Fractura', 'Gripe', np.nan],
    'Tratamiento': ['Antibiótico', 'Hidratación', 'Pomada', 'Antibiótico', 'Reposo',
                    'Antivirales', 'Antibiótico']
})

df_vet['Peso_original'] = df_vet['Peso']  # guardamos una copia para poder comparar "antes/después"
df_vet


In [ ]:
# --- Imputamos con SimpleImputer ---
# SimpleImputer necesita un array 2D, por eso pasamos df_vet[['Peso']] con doble corchete.

# Peso y Edad_Estimada son numéricas -> mediana (robusta a valores extremos)
imputer_mediana = SimpleImputer(strategy='median')
df_vet['Peso'] = imputer_mediana.fit_transform(df_vet[['Peso']])
df_vet['Edad_Estimada'] = imputer_mediana.fit_transform(df_vet[['Edad_Estimada']])

# Diagnostico es categórica -> moda (most_frequent)
imputer_moda = SimpleImputer(strategy='most_frequent')
df_vet['Diagnostico'] = imputer_moda.fit_transform(df_vet[['Diagnostico']]).ravel()
# .ravel() aplana el resultado (que sale como matriz 2D) a una lista simple de 1 columna

df_vet


In [ ]:
# --- Comparamos la distribución del Peso antes y después de imputar ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_vet['Peso_original'].dropna(), kde=True, color='red', bins=10)
plt.title('Distribución de Peso (original, sin imputar)')

plt.subplot(1, 2, 2)
sns.histplot(df_vet['Peso'], kde=True, color='green', bins=10)
plt.title('Distribución de Peso (imputado con mediana)')

plt.tight_layout()
plt.show()


<a name='seccion5'></a>
# 5. Bloque 4 Práctico – `Limpieza_Nulos_Parte2.ipynb`

**Consigna:** dataset *Titanic*.
1. Imputar la **edad** con la mediana **por clase de pasajero** (`groupby` + `transform`).
2. Imputar el **puerto de embarque** con el valor más frecuente.
3. Imputar las **tarifas** faltantes con `SimpleImputer(strategy='median')`.
4. Comparar antes/después con histogramas.

> ℹ️ Nota de nombres de columnas: `seaborn.load_dataset('titanic')` trae las columnas en minúscula (`age`, `pclass`, `embarked`, `fare`), a diferencia del CSV original de Kaggle que las trae en mayúscula (`Age`, `Pclass`, `Embarked`, `Fare`). Usamos las de seaborn porque son las que se cargan con una sola línea, sin depender de subir un archivo.

In [ ]:
# Cargamos el dataset Titanic que viene incluido en seaborn (no hace falta descargar nada)
df_titanic = sns.load_dataset('titanic')

df_titanic.info()


In [ ]:
# Guardamos una copia de la edad original ANTES de tocarla, para poder comparar después
df_titanic['age_original'] = df_titanic['age']

# --- Ejercicio 1: Imputar 'age' con la mediana POR CLASE ('pclass') ---
# groupby('pclass')['age'] agrupa las edades según la clase del pasajero (1, 2 o 3).
# transform(...) calcula la mediana de cada grupo y la usa para rellenar los NaN de ESE grupo.
df_titanic['age'] = df_titanic.groupby('pclass')['age'].transform(lambda x: x.fillna(x.median()))

print("Nulos en 'age' después de imputar:", df_titanic['age'].isnull().sum())


In [ ]:
# --- Ejercicio 2: Imputar 'embarked' (puerto de embarque) con el valor más frecuente ---
# .mode()[0] devuelve el valor que más se repite en la columna (puede haber más de una moda,
# por eso tomamos el primero con [0]).
df_titanic['embarked'] = df_titanic['embarked'].fillna(df_titanic['embarked'].mode()[0])

print("Nulos en 'embarked' después de imputar:", df_titanic['embarked'].isnull().sum())


In [ ]:
# --- Ejercicio 3: Imputar 'fare' (tarifa) con SimpleImputer(strategy='median') ---
imputer_fare = SimpleImputer(strategy='median')

# fit_transform espera un array 2D, por eso usamos doble corchete: df_titanic[['fare']]
df_titanic['fare'] = imputer_fare.fit_transform(df_titanic[['fare']])

print("Nulos en 'fare' después de imputar:", df_titanic['fare'].isnull().sum())


In [ ]:
# --- Ejercicio 4: Comparar histogramas de 'age' antes y después de imputar ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_titanic['age_original'].dropna(), kde=True, color='red', bins=30)
plt.title('Distribución de Edad (original)')

plt.subplot(1, 2, 2)
sns.histplot(df_titanic['age'], kde=True, color='green', bins=30)
plt.title('Distribución de Edad (imputada por clase)')

plt.tight_layout()
plt.show()

# Nota: si la imputación "aplana" mucho el histograma verde respecto al rojo (un pico muy marcado
# en la mediana), es una señal de que estamos perdiendo variabilidad real — justo el riesgo del
# que habla la teoría del Bloque 3.


In [ ]:
# --- Verificación final: ya no debería quedar ningún nulo en las columnas trabajadas ---
df_titanic[['age', 'embarked', 'fare']].isnull().sum()


<a name='seccion6'></a>
# 6. Bonus – Detección rápida de outliers

No lo pide explícitamente la consigna, pero como en el dataset de productos (`df_house`) metimos outliers a propósito (Ventas=50000, Precio_Unitario=500, Stock=10000), vale la pena mostrarlos: un boxplot es la forma más rápida de detectarlos visualmente. Esto conecta con la teoría: **imputar con la media** en una columna con outliers es peligroso porque la media se "va" hacia el outlier; por eso en todo este notebook usamos **mediana**, que no se ve afectada de la misma forma.

In [ ]:
# Recargamos el CSV original (antes de la limpieza) para ver los outliers "crudos"
df_house_crudo = pd.read_csv('data_imputacion.csv')

plt.figure(figsize=(12, 4))
for i, col in enumerate(['Ventas', 'Precio_Unitario', 'Stock'], start=1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df_house_crudo[col], color='skyblue')
    plt.title(col)
plt.tight_layout()
plt.show()

# En un boxplot, los puntitos sueltos por fuera de los "bigotes" son los outliers.


<a name='seccion7'></a>
# 7. Conclusión y contenido sugerido para el `README.md` del entregable

La consigna (Entregable 4) pide un `.zip` con `Limpieza_Nulos_Parte1.ipynb`, `Limpieza_Nulos_Parte2.ipynb` y un `README.md` explicando qué técnica se usó en cada caso y por qué. Si separás este notebook en dos archivos (Sección 3 → Parte1, Sección 5 → Parte2), podés usar este texto como base para el README:

```
README.md

Dataset 1 (equivalente a House Prices) – Limpieza_Nulos_Parte1.ipynb
----------------------------------------------------------------------
- Columnas eliminadas si tenían más del 50% de nulos (dropna con thresh).
- Filas sin Precio_Unitario (columna clave) eliminadas directamente,
  porque sin esa variable la fila no aporta al análisis.
- Numéricas (Ventas, Stock): imputadas con la MEDIANA, porque el dataset
  tiene outliers y la mediana no se distorsiona con valores extremos
  (a diferencia del promedio).
- Categóricas (Categoria, Region): imputadas con la MODA (valor más
  frecuente), porque no existe un "promedio" de una categoría de texto.

Dataset 2 (Titanic) – Limpieza_Nulos_Parte2.ipynb
----------------------------------------------------------------------
- Edad (age): imputada con la MEDIANA POR CLASE (pclass), no con una
  mediana global, porque la edad esperada varía bastante según la clase
  del pasajero (ej: 1ra clase tendía a tener pasajeros de más edad).
  Esto es imputación condicional / por grupo.
- Puerto de embarque (embarked): imputado con la MODA, ya que es una
  variable categórica con muy pocos valores posibles y casi todos los
  pasajeros embarcaron en el mismo puerto.
- Tarifa (fare): imputada con SimpleImputer(strategy='median'), por la
  misma razón que Ventas/Stock: es una variable con outliers (pasajes
  muy caros de 1ra clase).
```

**Para entregar como pide la consigna:** copiá la Sección 0 (imports) + Sección 3 completa en un notebook nuevo llamado `Limpieza_Nulos_Parte1.ipynb`, y la Sección 0 + Sección 5 en otro llamado `Limpieza_Nulos_Parte2.ipynb`. Las Secciones 1, 2, 4 y 6 son ejercicios complementarios/de práctica que hiciste en clase, no forman parte estricta del entregable.